<a href="https://colab.research.google.com/github/MBR4V0/Python/blob/main/Glue-PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**DynamicFrame do Glue em um DataFrame do PySpark**

In [ ]:
import sys
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql.functions import col, when

args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args['JOB_NAME'], args)

# 1. Lê os dados brutos do S3
df_bruto = glueContext.create_dynamic_frame.from_options(
    connection_type="s3",
    connection_options={"paths": ["s3://meu-bucket/vendas_brutas/"]},
    format="json"
).toDF() # Convertendo para PySpark DataFrame standard

# 2. Aplica as transformações com PySpark
df_limpo = df_bruto \
    .filter(col("status_pedido") == "CONCLUIDO") \
    .withColumnRenamed("id_cliente", "id_usuario") \
    .fillna({"valor_total": 0.0}) \
    .select("id_pedido", "id_usuario", "valor_total", "data_venda")

# 3. Converte de volta para DynamicFrame e salva particionado por Data
from awsglue.dynamicframe import DynamicFrame
dynamic_frame_final = DynamicFrame.fromDF(df_limpo, glueContext, "dynamic_frame_final")

glueContext.write_dynamic_frame.from_options(
    frame = dynamic_frame_final,
    connection_type = "s3",
    connection_options = {
        "path": "s3://meu-bucket/vendas_processadas/",
        "partitionKeys": ["data_venda"] # Particiona os dados no S3 para acelerar consultas futuras
    },
    format = "parquet"
)

job.commit()